# Next-day direction and range: LightGBM and a GRU

The method of [`docs/ml_plan.md`](../docs/ml_plan.md) run end to end, on
[`src/tools/ml_models/`](../src/tools/ml_models/): bars, features, labels, walk-forward folds,
baselines, metrics, both models and the sanity checks all come from the package, as the tests
check them.

| Section | Question |
|---|---|
| 1 Bars, targets, features | What does each model see at the close of day t, and what must it forecast for t+1? Does any feature look ahead? |
| 2 Walk-forward | How are the folds cut, and what is purged at their boundaries? |
| 3 Baselines | What does a model have to beat? |
| 4 LightGBM | One `binary` model for the direction; one `quantile` model per τ for the range. |
| 5 GRU | The same two targets from a 60-day window, with three input sets to compare. |
| 6 Results and sanity checks | On the synthetic tickers, is the direction no better than chance, and does the range beat a constant-width band? |
| 7 The checks can fail | A planted signal must be found; two leaks must be caught; a forecast must not change when later bars are removed. |

**Data.** The default is the six synthetic tickers (generated, offline, not market data). To run
on real prices, save them first with `save_all()` or the dashboard's **Download all**, then set
`DATA = 'local'` and list the symbols. **Do not commit this notebook with outputs**: run on saved
data, its tables and charts would be market data, and a test refuses a committed ML notebook
with outputs. Clear all outputs before committing.

**Needs** the `stats` and `ml` extras: `pip install -e ".[stats,ml]"` (for a CPU-only PyTorch,
first `pip install torch --index-url https://download.pytorch.org/whl/cpu`). The whole run takes
about 10 minutes on four CPU cores, most of it the GRU's three input sets; `GRU_INPUTS` and
`GRU_TICKERS` limit what it runs.

In [ ]:
# ── Environment setup: Google Colab vs local ────────────────────────
# Locally the project is installed once from the repo root with `pip install -e ".[stats,ml]"`,
# so `from src.tools...` resolves from any working directory. On Colab, an editable install is
# only seen by a new interpreter, so the repo root goes on sys.path for this kernel.
import os
import sys

REPO = '/content/drive/MyDrive/github/quant_dev'

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    if not os.path.isdir(REPO):
        raise FileNotFoundError(f'No repo at {REPO}; edit REPO to point at your clone.')
    %pip install -q -e "{REPO}[stats,ml]"
    sys.path.insert(0, REPO)

In [ ]:
import os
import time

import numpy as np
import pandas as pd
from IPython.display import display

from src.tools import ml_models as ml
from src.tools.price_return import synthetic_tickers

try:
    import lightgbm
    import torch
except ImportError as exc:
    raise ImportError('This notebook needs the ml extra: pip install -e ".[stats,ml]"') from exc

print(f'LightGBM {lightgbm.__version__}, PyTorch {torch.__version__}')
pct = lambda frame, cols, digits=1: frame.style.format({c: f'{{:.{digits}%}}' for c in cols})

## Parameters

The design choices confirmed in the plan (2026-10-02) are the package's defaults: binary
direction, next-close quantiles at `ml.TAUS`, an expanding walk-forward window of 756 rows
retrained every 63. `LONG` are the tickers the sanity checks judge: SYN-LEV starts in August
2022, which leaves too few test days after the 3-year warm-up, so it is shown but not judged.

In [ ]:
DATA = 'synthetic'                       # 'synthetic' (offline) or 'local' (saved Yahoo data)
TICKERS = synthetic_tickers() if DATA == 'synthetic' else ['ES=F', 'NQ=F', 'GC=F', 'CL=F']
LONG = [t for t in TICKERS if t != 'SYN-LEV']       # judged by the sanity checks
GRU_TICKERS = TICKERS                    # the GRU is the slow part; shorten this list to save time
GRU_INPUTS = list(ml.GRU_INPUTS)         # 'returns_range' (the default), 'returns', 'features'
GRU_STEP = 252                           # retrain every 252 rows; the plan's 63 takes four times as long
N_JOBS = None                            # LightGBM tickers in parallel; None is one per CPU
WINDOW = 60                              # the GRU's look-back, in bars

## 1 Bars, targets, features

`C_t` is the close of bar t. The only forward-looking columns are the two labels: tomorrow's
direction `y_up = 1[C_{t+1} > C_t]` and tomorrow's return `y_ret = r_{t+1}`, whose quantiles the
range models forecast. A return that spans a non-positive close (SYN-OIL on 2020-04-20) is
dropped, as `price_return` drops it. The 16 features (`ml.FEATURES`) use bar t and the bars
before it, and none depends on the price level.

In [ ]:
BARS = {t: ml.ticker_bars(t, source=DATA) for t in TICKERS}
DATA_BY_TICKER = {t: ml.dataset(b) for t, b in BARS.items()}
pd.DataFrame({t: {'rows': len(d), 'from': d.index[0].date(), 'to': d.index[-1].date(),
                  'up share': f"{d['y_up'].mean():.1%}"} for t, d in DATA_BY_TICKER.items()}).T

**No look-ahead.** The test `ta_tools` uses for its indicators: cut the bars after t, recompute,
and every feature at or before t must be unchanged. A feature that peeked at a later bar would
change. (`tests/test_ml_models.py` runs this per feature; here it is shown once.)

In [ ]:
def check_no_lookahead(bars, cuts=(300, 1180, 2000)):
    full = ml.features(bars)
    for cut in cuts:
        pd.testing.assert_frame_equal(ml.features(bars.iloc[:cut + 1]), full.iloc[:cut + 1],
                                      check_exact=False, rtol=1e-12)
    return f'{len(ml.FEATURES)} features unchanged at {len(cuts)} cut points'

check_no_lookahead(BARS[TICKERS[0]])

## 2 Walk-forward

Expanding window: the first fold trains on the first 756 rows and tests the next 63; each later
fold adds 63 rows to the training window. A training row whose label interval (t, t+h] reaches
the first test row is purged: h - 1 rows, so none for the next-day labels. Early stopping holds
out the last 20% of each training window, purged the same way.

In [ ]:
for h in (1, 5):
    train, test = ml.walk_forward(3000, horizon=h)[0]
    print(f'h = {h}: last training row {train[-1]}, its label reaches {train[-1] + h}, '
          f'first test row {test[0]}: {test[0] - train[-1] - 1} purged')
print(f'{TICKERS[0]}: {len(ml.walk_forward(len(DATA_BY_TICKER[TICKERS[0]])))} folds')

## 3 Baselines

Each is refitted per fold from the training rows, or uses only what is known at t:
the base rate for the direction; for the range, a constant-width band (the training quantiles),
`price_range`'s lognormal ±zσ band on the 250-day volatility, a 20-day σ band, and the EWMA
volatility times the training quantiles of `r_{t+1} / σ_t`. The last is the one to beat: on
fat-tailed returns the normal-quantile bands over-cover, and the constant band's misses cluster.

In [ ]:
rows = []
for t in LONG:
    d = DATA_BY_TICKER[t]
    bands = ml.range_baselines(d, ml.walk_forward(len(d)))
    y = d.loc[bands.index, 'y_ret']
    for band in ml.BANDS:
        s, _ = ml.range_scores({tau: bands[(band, tau)] for tau in ml.TAUS}, y)
        rows.append({'ticker': t, 'band': band, 'cover 68%': s['cover_68'], 'cover 90%': s['cover_90'],
                     'miss after miss / after hit': s['cluster_ratio_68'], 'pinball (bp)': s['pinball'] * 1e4})
BASELINES = pd.DataFrame(rows).set_index(['ticker', 'band'])
pct(BASELINES, ['cover 68%', 'cover 90%']).format({'miss after miss / after hit': '{:.2f}', 'pinball (bp)': '{:.2f}'})

## 4 LightGBM

`ml.LightGBMModel`: 15 leaves, at least 50 rows a leaf, learning rate 0.03, up to 300 trees,
early-stopped on the purged tail and refitted on the whole training window, deterministic on one
thread. The range models fit `r_{t+1} / σ_t` and scale back by σ_t. `ml.evaluate_many` runs the
tickers in parallel processes; the forecasts are the same as one at a time.

In [ ]:
t0 = time.time()
LGB = {fc.ticker: fc for fc in ml.evaluate_many(TICKERS, ml.LightGBMModel(), source=DATA, n_jobs=N_JOBS)}
print(f'LightGBM walk-forward on {len(TICKERS)} tickers: {time.time() - t0:.0f} s')
pd.DataFrame({t: {'folds': len(fc.folds), 'median trees (direction)': int(fc.folds['direction_trees'].median()),
                  'top features (direction)': ', '.join(ml.feature_importance(fc).index[:3])}
              for t, fc in LGB.items()}).T

In [ ]:
ml.plot_folds(LGB[TICKERS[0]]).show()
ml.plot_feature_importance(ml.feature_importance(LGB[TICKERS[0]]),
                           title=f'{TICKERS[0]}: direction model, share of gain per feature').show()

## 5 GRU

`ml.GRUModel`: one GRU layer of 32 units over the last 60 rows of its inputs, standardized with
the training window's means and standard deviations. The direction network ends in one logit;
the range network in five quantiles of the standardized return, scaled back by σ_t and
**recalibrated**: each τ is shifted by the τ-quantile of the network's own errors on the
validation tail, which keeps the intervals from being too narrow. Early stopping on the purged
tail, seeded per fold, deterministic on CPU.

**Three input sets, kept until a model is trained on real data** (the owner's choice):
`'returns_range'` (each day's return and its true range, the provisional default), `'returns'`
(the return alone) and `'features'` (the 16 features LightGBM reads). On about 1,600 training rows
the 16 features over 60 days are too many inputs: that GRU finds no planted signal (section 7).
To save time the GRU retrains every `GRU_STEP` rows, over the same test days as LightGBM.

In [ ]:
GRU = {}
t0 = time.time()
for inputs in GRU_INPUTS:
    GRU[inputs] = {fc.ticker: fc for fc in ml.evaluate_many(
        GRU_TICKERS, ml.GRUModel(features=inputs, threads=1), source=DATA, n_jobs=N_JOBS, step=GRU_STEP)}
    print(f'GRU ({inputs}): {time.time() - t0:.0f} s')

## 6 Results and sanity checks

**Direction**: log loss and Brier of the model against the base rate, with the 99% bootstrap
interval of `log loss(base) - log loss(model)` (positive means the model is better). On the
synthetic tickers the sign of tomorrow's return is independent of the past by construction, so
the model must be no better than the base rate: the interval must not lie wholly above zero,
and the mean gain must be at most 0.002 (`no_better_than_chance`).

**Range**: coverage, Kupiec, Christoffersen and pinball loss for the model and every baseline.
The model must have a lower pinball loss than the constant band on each long ticker, misses that
cluster less, and a pooled gain whose 95% interval lies wholly above zero.

In [ ]:
DIRECTION_COLS = ['days', 'log_loss_base', 'log_loss_model', 'gain', 'lower', 'upper',
                  'forecast_spread', 'no_better_than_chance']


def results(name, forecasts):
    direction = pd.DataFrame([ml.direction_scores(fc) for fc in forecasts.values()])[DIRECTION_COLS]
    display(direction.style.format({c: '{:+.4f}' if c in ('gain', 'lower', 'upper') else '{:.4f}'
                                    for c in DIRECTION_COLS[1:-1]}).set_caption(f'{name}: direction'))
    ranges = pd.concat({t: ml.range_table(fc).loc[['constant', 'ewma_std_q', 'model'],
                                                 ['cover_68', 'cover_90', 'kupiec_p_68',
                                                  'christoffersen_p_68', 'cluster_ratio_68', 'pinball']]
                        for t, fc in forecasts.items()})
    ranges['pinball'] *= 1e4
    display(ranges.style.format({'cover_68': '{:.1%}', 'cover_90': '{:.1%}', 'kupiec_p_68': '{:.3f}',
                                 'christoffersen_p_68': '{:.2g}', 'cluster_ratio_68': '{:.2f}',
                                 'pinball': '{:.2f}'}).set_caption(f'{name}: range (pinball in bp)'))
    table, pooled, passes = ml.range_check([forecasts[t] for t in LONG if t in forecasts])
    display(table)
    chance = bool(direction.loc[direction.index.isin(LONG), 'no_better_than_chance'].all())
    print(f'{name}: direction no better than chance on every long ticker: {chance}')
    print(f'{name}: range pooled gain over the constant band {pooled["estimate"]:.1%} '
          f'(95% interval {pooled["lower"]:.1%} to {pooled["upper"]:.1%}); range checks pass: {passes}')
    return {'direction no better than chance': chance, 'range beats the constant band': passes,
            'pooled pinball gain': f'{pooled["estimate"]:.1%} ({pooled["lower"]:.1%} to {pooled["upper"]:.1%})'}


SUMMARY = {'LightGBM': results('LightGBM', LGB)}

In [ ]:
for inputs, forecasts in GRU.items():
    SUMMARY[f'GRU ({inputs})'] = results(f'GRU ({inputs})', forecasts)

### Charts

Reliability (marker area grows with the days in the bin), rolling coverage of the 68% interval,
the model's intervals through SYN-INDEX's scheduled sell-off, and every band's pinball loss
against the constant one.

In [ ]:
t = TICKERS[0]
ml.plot_reliability(LGB[t]).show()
ml.plot_coverage(LGB[t]).show()
ml.plot_forecast_bands(LGB[t], BARS[t], '2020-02-01', '2020-04-30').show()
ml.plot_range_comparison([LGB[t] for t in LONG]).show()
for inputs, forecasts in GRU.items():
    ml.plot_range_comparison([forecasts[t] for t in LONG if t in forecasts]).update_layout(
        title=f'GRU ({inputs}): mean pinball loss relative to the constant-width band').show()

## 7 The checks can fail

A check that cannot fail proves nothing; a model that always predicted the base rate would pass
the direction check trivially. So four controls on LightGBM, and the first for each GRU input set:

1. **A planted signal.** SYN-INDEX with each day's sign made to repeat the last 60% of the time
   (sizes and volatility kept). The direction model must now beat the base rate.
2. **A leaked future return.** `r_{t+1}` added as a feature. The direction check must fail.
3. **A centred window.** A 5-day average centred on t, a realistic look-ahead bug: it carries
   `r_{t+1}` and `r_{t+2}`. The direction check must fail.
4. **A forecast cannot see past its day.** Remove every bar after a test date (keeping the next
   close, which is only that date's label), refit that fold, and the forecast must be identical.

In [ ]:
model, leaky = ml.LightGBMModel(), ml.LightGBMModel(features=ml.FEATURES + ['leak'], max_trees=60)
index_data = ml.dataset(ml.ticker_bars('SYN-INDEX'))
r = ml.bar_returns(ml.ticker_bars('SYN-INDEX'))
controls = {
    'planted signal': ml.walk_forward_forecasts(ml.dataset(ml.planted_signal_bars()), model, ranges=False),
    'leaked r(t+1)': ml.walk_forward_forecasts(index_data.assign(leak=index_data['y_ret']), leaky, ranges=False),
    'centred window': ml.walk_forward_forecasts(
        index_data.assign(leak=r.rolling(5, center=True).mean().reindex(index_data.index)).dropna(),
        leaky, ranges=False),
}
CONTROL = pd.DataFrame({k: ml.direction_scores(fc) for k, fc in controls.items()}).T
CONTROL['must'] = ['beat the base rate', 'fail the check', 'fail the check']
CONTROL[['days', 'log_loss_base', 'log_loss_model', 'gain', 'lower', 'upper',
         'no_better_than_chance', 'beats_base_rate', 'must']]

In [ ]:
# The planted signal for each GRU input set: with the 16 features, too many inputs for the data,
# the GRU does not find it; the raw daily sequence does.
planted = ml.dataset(ml.planted_signal_bars())
GRU_PLANTED = pd.DataFrame({
    f'GRU ({inputs})': ml.direction_scores(ml.walk_forward_forecasts(
        planted, ml.GRUModel(features=inputs), ranges=False, step=GRU_STEP))
    for inputs in GRU_INPUTS}).T
GRU_PLANTED[['days', 'log_loss_base', 'log_loss_model', 'gain', 'lower', 'upper', 'beats_base_rate']]

In [ ]:
bars = ml.ticker_bars('SYN-INDEX')
full = LGB['SYN-INDEX'] if 'SYN-INDEX' in LGB else ml.evaluate('SYN-INDEX', model)
for position in (900, 1700, 2400):
    date = index_data.index[position]
    d = ml.dataset(bars.iloc[:bars.index.get_loc(date) + 2])
    fold = next(f for f in ml.walk_forward(len(d)) if date in d.index[f[1]])
    cut = ml.walk_forward_forecasts(d, model, folds=[fold])
    same = (cut.direction.loc[date, 'p_model'] == full.direction.loc[date, 'p_model']
            and cut.bands.loc[date, 'model'].equals(full.bands.loc[date, 'model']))
    print(f'{date:%Y-%m-%d}: direction {full.direction.loc[date, "p_model"]:.6f}, '
          f'median {full.bands.loc[date, ("model", 0.5)]:+.6f}; later bars removed: identical {same}')

## Summary

In [ ]:
summary = pd.DataFrame(SUMMARY)
display(summary)
print(f"Controls (LightGBM): planted signal found: {bool(CONTROL.loc['planted signal', 'beats_base_rate'])}; "
      f"leaked r(t+1) caught: {not CONTROL.loc['leaked r(t+1)', 'no_better_than_chance']}; "
      f"centred window caught: {not CONTROL.loc['centred window', 'no_better_than_chance']}")
for name, row in GRU_PLANTED.iterrows():
    print(f"{name}: planted signal found: {bool(row['beats_base_rate'])} (gain {row['gain']:+.4f})")